# Qwen-3B Full-Stack Pipeline (Colab)

**Goal:** produce the Colab-side artifacts needed to push Qwen-3B beyond the 100 dec_tps target on the laptop stack.

This notebook now has two jobs:

1. **Fix/reliability path:** pull a fresh repo every run, reuse the shared Drive corpus, resume calibration safely, and fail fast when required artifacts are stale or missing.
2. **Frontier path:** train a wider Eagle5 grid, train bounded residual-delta variants, run real tau eval, then search custom variable-K / entropy-routed / draft-lattice policies.

| Stage | What changed |
|---|---|
| Corpus | Uses `/MyDrive/dismantle/qwen3b_corpus` so it reuses the mega-calibration notebook instead of duplicating data under artifacts. |
| Calibration | Local SSD writes + Drive sync, GPU-adaptive batch/chunk, safe resume from shards + activation stats. |
| Training | Larger LR grid plus residual-delta-loss variants that make `draft_hidden` track the next residual state. |
| Tau Eval | Real Colab tau eval via `colab/eagle5_tau_eval_pytorch.py`; no placeholder JSON. |
| Frontier Search | `colab/eagle5_frontier_policy.py` searches variable-K confidence gates, entropy/margin routing, and tiny draft-lattice upper bounds. |
| Runtime Hints | Writes policy JSON with concrete env-style knobs for the next Qwen-specific hot-path wiring pass. |

Default mode is `MAX_TPS_MODE=True`. It is intentionally aggressive, but bounded for Colab Pro: the expensive frontier search runs only on the top tau candidates.


In [ ]:
# Cell 1 — Mount Drive, verify GPU, configure the run
from google.colab import drive
drive.mount('/content/drive')

import glob, json, math, os, shutil, subprocess, sys, time
from pathlib import Path

import torch
assert torch.cuda.is_available(), 'No CUDA — set Runtime → Change runtime type → GPU'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')

# Flip this to False for a quick smoke run. True is the path aimed at >100 dec_tps.
MAX_TPS_MODE = True
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
CAPTURE_LAYER = 32
BASELINE_DEC_TPS = 26.6
W4A8_MULTIPLIER = 1.25
SPEC_EFFICIENCY = 0.85

DRIVE_ROOT = '/content/drive/MyDrive/dismantle'
ARTIFACT_DIR = f'{DRIVE_ROOT}/qwen3b_artifacts'
CORPUS_DIR = f'{DRIVE_ROOT}/qwen3b_corpus'
LOCAL_CORPUS = '/content/qwen3b_corpus'

# 10k is the safer max-TPS default. If your other Colab is already building a
# bigger corpus, this notebook reuses it and skips once enough shards/stats exist.
CORPUS_TARGET_SEQS = 10_000 if MAX_TPS_MODE else 2_000
TRAIN_MAX_ROWS = 8_000 if MAX_TPS_MODE else 4_000
TRAIN_EPOCHS = 10 if MAX_TPS_MODE else 8
TRAIN_MAX_ROW_TOKENS = 192 if MAX_TPS_MODE else 128
EVAL_MAX_WINDOWS = 6_000 if MAX_TPS_MODE else 2_000
FRONTIER_MAX_DEPTH = 12 if MAX_TPS_MODE else 6
FRONTIER_TOP_N = 3 if MAX_TPS_MODE else 1
LRs = ([1e-4, 2e-4, 3e-4, 6e-4, 1e-3, 2e-3, 3e-3]
       if MAX_TPS_MODE else [3e-4, 1e-3, 3e-3])

# Bounded frontier variants: residual-delta loss trains the same head to make
# draft_hidden closer to the next residual state. This is the cheap Colab-Pro
# version of a residual simulator without adding new runtime tensor shapes.
TRAIN_SPECS = [{'lr': lr, 'residual_delta': 0.0, 'calib_weight': 0.10, 'family': 'base'} for lr in LRs]
if MAX_TPS_MODE:
    TRAIN_SPECS += [
        {'lr': 3e-4, 'residual_delta': 0.01, 'calib_weight': 0.15, 'family': 'rd'},
        {'lr': 6e-4, 'residual_delta': 0.02, 'calib_weight': 0.20, 'family': 'rd'},
        {'lr': 1e-3, 'residual_delta': 0.02, 'calib_weight': 0.20, 'family': 'rd'},
    ]

for d in (DRIVE_ROOT, ARTIFACT_DIR, CORPUS_DIR, LOCAL_CORPUS):
    os.makedirs(d, exist_ok=True)

print(f'MAX_TPS_MODE={MAX_TPS_MODE}')
print(f'corpus: target={CORPUS_TARGET_SEQS} seqs  drive={CORPUS_DIR}')
print(f'train: rows={TRAIN_MAX_ROWS} epochs={TRAIN_EPOCHS} max_row_tokens={TRAIN_MAX_ROW_TOKENS}')
print(f'train specs: {TRAIN_SPECS}')
print(f'frontier: top_n={FRONTIER_TOP_N} max_depth={FRONTIER_MAX_DEPTH} eval_windows={EVAL_MAX_WINDOWS}')
print(f'artifacts: {ARTIFACT_DIR}')


In [ ]:
# Cell 2 — Install deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
], check=True)
import transformers, datasets, pyarrow, accelerate, safetensors
print(f'transformers={transformers.__version__} torch={torch.__version__}')
print(f'datasets={datasets.__version__} pyarrow={pyarrow.__version__} accelerate={accelerate.__version__}')


In [ ]:
# Cell 3 — Clone/update dismantle to the exact GitHub main used by this Colab link
REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'
if not os.path.exists('/content/dismantle'):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/dismantle'], check=True)
else:
    subprocess.run(['git', '-C', '/content/dismantle', 'fetch', 'origin', BRANCH, '--depth', '1'], check=True)
    subprocess.run(['git', '-C', '/content/dismantle', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/dismantle', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir('/content/dismantle')
print('repo:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
for path in [
    'colab/mega_calibrate.py',
    'colab/eagle5_train_pytorch.py',
    'colab/eagle5_tau_eval_pytorch.py',
    'colab/eagle5_frontier_policy.py',
    'colab/awq_w4a8_validate.py',
    'colab/build_qwen3b_frozen_hf.py',
    'tools/training/awq_calibrate.py',
]:
    assert os.path.exists(path), f'missing required file: {path}'
    print('✓', path)


In [ ]:
# Cell 4 — Corpus build / resume on local SSD with durable Drive sync
import numpy as np

STATS_FILE = f'{CORPUS_DIR}/per_site_activation_stats.npz'
MANIFEST = f'{CORPUS_DIR}/manifest.json'
required_shards = math.ceil(CORPUS_TARGET_SEQS / 16)
existing_shards = len(glob.glob(f'{CORPUS_DIR}/shard_*.parquet'))
stats_sequences = 0
if os.path.exists(STATS_FILE):
    try:
        with np.load(STATS_FILE) as z:
            if 'sequences_written' in z.files:
                stats_sequences = int(np.asarray(z['sequences_written']).item())
    except Exception as e:
        print(f'WARN: could not read stats file: {e}')
manifest_target = 0
if os.path.exists(MANIFEST):
    try:
        manifest_target = int(json.load(open(MANIFEST)).get('max_sequences', 0))
    except Exception as e:
        print(f'WARN: could not read manifest: {e}')

complete = existing_shards >= required_shards and os.path.exists(STATS_FILE) and stats_sequences >= CORPUS_TARGET_SEQS
print(f'corpus state: shards={existing_shards}/{required_shards} stats_sequences={stats_sequences} manifest_target={manifest_target}')

if complete:
    print(f'✓ Corpus is complete enough for this run: {existing_shards} shards, stats through {stats_sequences} seqs')
else:
    os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    if VRAM_GB >= 70:
        tier, load_4bit, batch, lm_head_chunk = 'G4/H100/Blackwell 70GB+', False, 8, 256
    elif VRAM_GB >= 35:
        tier, load_4bit, batch, lm_head_chunk = 'A100 40GB-class', False, 6, 128
    elif VRAM_GB >= 20:
        tier, load_4bit, batch, lm_head_chunk = 'L4 24GB-class', True, 4, 64
    else:
        tier, load_4bit, batch, lm_head_chunk = 'T4/V100 16GB-class', True, 2, 32
    print(f'Building/resuming corpus: tier={tier} batch={batch} lm_head_chunk={lm_head_chunk} load={"4-bit" if load_4bit else "fp16"}')
    cmd = [
        sys.executable, 'colab/mega_calibrate.py',
        '--model', MODEL_ID,
        '--max-sequences', str(CORPUS_TARGET_SEQS),
        '--max-tokens', '2048',
        '--batch-size', str(batch),
        '--capture-layer', str(CAPTURE_LAYER),
        '--topk-logits', '100',
        '--lm-head-chunk-tokens', str(lm_head_chunk),
        '--shard-size', '16',
        '--sync-dir', CORPUS_DIR,
        '--sync-every', '4',
        '--delete-local-after-sync',
        '--out', LOCAL_CORPUS,
    ]
    if load_4bit:
        cmd.append('--load-4bit')
    subprocess.run(cmd, check=True)
    existing_shards = len(glob.glob(f'{CORPUS_DIR}/shard_*.parquet'))
    print(f'✓ Corpus now has {existing_shards} Drive shards')


In [ ]:
# Cell 5 — AWQ smoothing factors from the latest activation stats
AWQ_OUT = f'{ARTIFACT_DIR}/qwen3b_awq_smoothing.json'
assert os.path.exists(STATS_FILE), f'missing activation stats: {STATS_FILE}'
subprocess.run([
    sys.executable, 'tools/training/awq_calibrate.py',
    '--stats', STATS_FILE,
    '--out', AWQ_OUT,
    '--alpha', '0.5',
    '--model', MODEL_ID,
], check=True)
with open(AWQ_OUT) as f:
    awq = json.load(f)
print(f'AWQ: {len(awq["smoothing_factors"])} factors, sequences={awq.get("sequences_calibrated")}, mean={awq["stats"]["mean_factor"]:.4f}, max={awq["stats"]["max_factor"]:.2f}')


In [ ]:
# Cell 6 — Qwen-3B frozen baseline for Eagle5 training
FROZEN_OUT = f'{ARTIFACT_DIR}/qwen3b_frozen.npz'
subprocess.run([
    sys.executable, 'colab/build_qwen3b_frozen_hf.py',
    '--model', MODEL_ID,
    '--out', FROZEN_OUT,
], check=True)
assert os.path.exists(FROZEN_OUT), f'missing frozen baseline: {FROZEN_OUT}'
print(f'✓ frozen.npz ready ({os.path.getsize(FROZEN_OUT)/1e9:.2f} GB)')


In [ ]:
# Cell 7 — Eagle5 PyTorch LR + residual-delta race
TRAIN_BATCH = 32 if VRAM_GB >= 35 else 16
COMPILE_FLAG = '--compile' if (MAX_TPS_MODE and VRAM_GB >= 35) else ''
FORCE_RETRAIN = False

def spec_tag(spec):
    lr_tag = f"lr{spec['lr']:.0e}".replace('+', '').replace('-0', '-')
    rd = int(round(spec['residual_delta'] * 1000))
    cw = int(round(spec['calib_weight'] * 100))
    suffix = '' if spec['family'] == 'base' else f"_{spec['family']}rd{rd:03d}_cw{cw:02d}"
    return f'{lr_tag}{suffix}'

trained = []
for spec in TRAIN_SPECS:
    tag = spec_tag(spec)
    lr = spec['lr']
    ckpt_dir = f'{ARTIFACT_DIR}/checkpoints/eagle5_qwen3b_{tag}'
    final_head = f'{ckpt_dir}/head_final.safetensors'
    os.makedirs(ckpt_dir, exist_ok=True)
    if os.path.exists(final_head) and not FORCE_RETRAIN:
        print(f'✓ {tag} already trained: {final_head}')
        trained.append((tag, spec, final_head))
        continue
    print()
    print(f'=== Training variant {tag} spec={spec} → {ckpt_dir}')
    cmd = [
        sys.executable, 'colab/eagle5_train_pytorch.py',
        '--corpus-dir', CORPUS_DIR,
        '--frozen', FROZEN_OUT,
        '--ckpt-dir', ckpt_dir,
        '--epochs', str(TRAIN_EPOCHS),
        '--batch-size', str(TRAIN_BATCH),
        '--seq-len', '16',
        '--lr', str(lr),
        '--capture-layer', str(CAPTURE_LAYER),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--max-row-tokens', str(TRAIN_MAX_ROW_TOKENS),
        '--sparsity-head', 'off',
        '--seed', '0',
        '--calib-loss-weight', str(spec['calib_weight']),
        '--residual-delta-loss-weight', str(spec['residual_delta']),
        '--save-safetensors',
    ]
    if COMPILE_FLAG:
        cmd.append(COMPILE_FLAG)
    subprocess.run(cmd, check=True)
    assert os.path.exists(final_head), f'training did not write {final_head}'
    trained.append((tag, spec, final_head))
print(f'✓ trained/available variants: {[t[0] for t in trained]}')


In [ ]:
# Cell 8 — Real Colab tau-at-depth eval and first-pass winner selection
TAU_DEPTH = 4
tau_results = {}
for tag, spec, ckpt in trained:
    out_path = f'{ARTIFACT_DIR}/eagle5_tau_{tag}.json'
    print()
    print(f'=== τ-eval {tag}')
    subprocess.run([
        sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
        '--ckpt', ckpt,
        '--frozen', FROZEN_OUT,
        '--corpus', CORPUS_DIR,
        '--out', out_path,
        '--depth', str(TAU_DEPTH),
        '--max-windows', str(EVAL_MAX_WINDOWS),
        '--max-row-tokens', str(TRAIN_MAX_ROW_TOKENS),
        '--base-tps', str(BASELINE_DEC_TPS),
        '--w4a8-multiplier', str(W4A8_MULTIPLIER),
        '--spec-efficiency', str(SPEC_EFFICIENCY),
    ], check=True)
    with open(out_path) as f:
        tau_results[tag] = json.load(f)
    tau_results[tag]['train_spec'] = spec
    tau_results[tag]['ckpt'] = ckpt

ranked_tau = sorted(
    tau_results.items(),
    key=lambda kv: (kv[1]['projection']['projected_dec_tps'], kv[1]['tau'], kv[1]['depth1_accept_rate']),
    reverse=True,
)
winner_tag, winner = ranked_tau[0]
winner_path = f'{ARTIFACT_DIR}/eagle5_tau_winner.json'
with open(winner_path, 'w') as f:
    json.dump({'winner': winner_tag, 'result': winner, 'all': tau_results}, f, indent=2, sort_keys=True)

print()
print('τ race results:')
for tag, r in ranked_tau:
    print(f"  {tag:20s} tau={r['tau']:.3f} depth1={r['depth1_accept_rate']:.2%} full@{TAU_DEPTH}={r['full_accept_rate']:.2%} projected={r['projection']['projected_dec_tps']:.1f} dec_tps spec={r['train_spec']}")
print()
print(f'✓ tau_winner={winner_tag} projected={winner["projection"]["projected_dec_tps"]:.1f} dec_tps → {winner_path}')


In [ ]:
# Cell 9 — Frontier policy search: variable-K, entropy routing, lattice, residual probe
frontier_results = {}
frontier_candidates = ranked_tau[:FRONTIER_TOP_N]
for tag, r in frontier_candidates:
    out_path = f'{ARTIFACT_DIR}/eagle5_frontier_{tag}.json'
    print()
    print(f'=== frontier policy search {tag}')
    subprocess.run([
        sys.executable, 'colab/eagle5_frontier_policy.py',
        '--ckpt', r['ckpt'],
        '--frozen', FROZEN_OUT,
        '--corpus', CORPUS_DIR,
        '--out', out_path,
        '--max-depth', str(FRONTIER_MAX_DEPTH),
        '--depths', '4,6,8,12',
        '--lattice-widths', '2,3,4',
        '--max-windows', str(EVAL_MAX_WINDOWS),
        '--max-row-tokens', str(TRAIN_MAX_ROW_TOKENS),
        '--base-tps', str(BASELINE_DEC_TPS),
        '--w4a8-multiplier', str(W4A8_MULTIPLIER),
        '--spec-efficiency', str(SPEC_EFFICIENCY),
    ], check=True)
    with open(out_path) as f:
        frontier_results[tag] = json.load(f)

frontier_ranked = sorted(
    frontier_results.items(),
    key=lambda kv: kv[1]['policies']['best_deployable']['projected_dec_tps'],
    reverse=True,
)
frontier_winner_tag, frontier_winner = frontier_ranked[0]
frontier_winner_path = f'{ARTIFACT_DIR}/eagle5_frontier_winner.json'
with open(frontier_winner_path, 'w') as f:
    json.dump({'winner': frontier_winner_tag, 'result': frontier_winner, 'all': frontier_results}, f, indent=2, sort_keys=True)

print()
print('Frontier policy results:')
for tag, payload in frontier_ranked:
    best = payload['policies']['best_deployable']
    rd = payload['policies']['residual_delta_probe']
    print(f"  {tag:20s} deployable={best['kind']:34s} projected={best['projected_dec_tps']:.1f} accepted={best['accepted_draft_tokens_per_verify']:.2f} residual_cos={rd['cosine_mean']}")
print()
print(f'✓ frontier_winner={frontier_winner_tag} → {frontier_winner_path}')


In [ ]:
# Cell 10 — AWQ + W4A8 fake-quant validation
VALIDATION_OUT = f'{ARTIFACT_DIR}/awq_w4a8_validation.json'
if VRAM_GB < 20:
    print('Skipping W4A8 validation on <20 GB GPU; fake-quant validator needs fp16 Linears to instrument.')
else:
    subprocess.run([
        sys.executable, 'colab/awq_w4a8_validate.py',
        '--model', MODEL_ID,
        '--smoothing', AWQ_OUT,
        '--out', VALIDATION_OUT,
        '--n-prompts', '30',
        '--max-new-tokens', '32',
        '--device', 'cuda',
    ], check=True)
    with open(VALIDATION_OUT) as f:
        d = json.load(f)
    print()
    print('AWQ validation:')
    print(f'  WITH AWQ:    KL={d["with_awq"]["mean_kl_per_token"]:.4f}  match@8={d["with_awq"]["greedy_match_first_8"]:.2%}  match@32={d["with_awq"]["greedy_match_first_32"]:.2%}')
    print(f'  WITHOUT AWQ: KL={d["without_awq"]["mean_kl_per_token"]:.4f}  match@8={d["without_awq"]["greedy_match_first_8"]:.2%}  match@32={d["without_awq"]["greedy_match_first_32"]:.2%}')
    print(f'  Δ            KL_improvement={d["delta"]["kl_improvement"]:.4f}  match_improvement={d["delta"]["match_improvement"]:+.2%}')


In [ ]:
# Cell 11 — Summary + how this gets past 100 TPS
frontier_info_path = f'{ARTIFACT_DIR}/eagle5_frontier_winner.json'
tau_info_path = f'{ARTIFACT_DIR}/eagle5_tau_winner.json'
frontier_info = json.load(open(frontier_info_path)) if os.path.exists(frontier_info_path) else None
tau_info = json.load(open(tau_info_path)) if os.path.exists(tau_info_path) else None
summary_lines = [
    '# Qwen-3B Full-Stack Pipeline — Summary',
    '',
    f'Generated by `colab/qwen3b_full_stack.ipynb` on {GPU_NAME}.',
    '',
    '## Run mode',
    '',
    f'- MAX_TPS_MODE: `{MAX_TPS_MODE}`',
    f'- corpus target: `{CORPUS_TARGET_SEQS}` sequences',
    f'- train rows: `{TRAIN_MAX_ROWS}`',
    f'- train epochs: `{TRAIN_EPOCHS}`',
    f'- train specs: `{TRAIN_SPECS}`',
    '',
    '## Artifacts',
    '',
]
for f in ['qwen3b_frozen.npz', 'qwen3b_awq_smoothing.json', 'awq_w4a8_validation.json', 'eagle5_tau_winner.json', 'eagle5_frontier_winner.json']:
    path = f'{ARTIFACT_DIR}/{f}'
    if os.path.exists(path):
        summary_lines.append(f'- `{path}` ({os.path.getsize(path)/1e6:.1f} MB)')
if os.path.isdir(CORPUS_DIR):
    corpus_size = sum(os.path.getsize(os.path.join(r, fn)) for r, _, fns in os.walk(CORPUS_DIR) for fn in fns) / 1e9
    summary_lines.append(f'- `{CORPUS_DIR}/` ({corpus_size:.2f} GB)')

summary_lines.extend(['', '## Tau Candidates', ''])
for tag, r in ranked_tau:
    summary_lines.append(f"- `{tag}`: tau@4={r['tau']:.3f}, depth1={r['depth1_accept_rate']:.2%}, projected={r['projection']['projected_dec_tps']:.1f} dec_tps, spec={r['train_spec']}")

if frontier_info:
    wtag = frontier_info['winner']
    best = frontier_info['result']['policies']['best_deployable']
    hints = frontier_info['result']['runtime_hints']
    summary_lines.extend([
        '',
        '## Frontier Winner',
        '',
        f'`{wtag}` wins the frontier policy search.',
        f'- best deployable policy: `{best["kind"]}`',
        f'- accepted draft tokens/verify: `{best["accepted_draft_tokens_per_verify"]:.3f}`',
        f'- projected deployable stack: `{best["projected_dec_tps"]:.1f}` dec_tps',
        '',
        'Runtime hints for the next Rust wiring pass:',
        '',
        '```json',
        json.dumps(hints, indent=2, sort_keys=True),
        '```',
    ])
    wtag_for_path = wtag
elif tau_info:
    wtag_for_path = tau_info['winner']
else:
    wtag_for_path = 'WINNER_TAG'

summary_lines.extend([
    '',
    '## What Was Added For The Paradigm Push',
    '',
    '- **Variable-K Eagle**: searched confidence thresholds from the calibration head instead of fixed K only.',
    '- **Entropy/margin routing**: searches when to skip speculation on chaotic positions using full-model margin.',
    '- **Draft lattice upper bound**: measures tiny top-2/top-3/top-4 branch coverage, but does not rank it as deployable until tree verification is wired.',
    '- **Residual-delta simulator training**: selected variants add a next-residual alignment loss without changing runtime tensor shapes.',
    '- **Static profile hints**: policy JSON emits concrete env-style knobs for a Qwen-specific hot path.',
    '',
    '## Local next step',
    '',
    f'Download `{ARTIFACT_DIR}` and `{CORPUS_DIR}` from Drive, then run the paired bench with the winner head:',
    '',
    '```bash',
    f'EAGLE5_HEAD=artifacts/qwen3b_artifacts/checkpoints/eagle5_qwen3b_{wtag_for_path}/head_final.safetensors \\',
    '  TRIALS=10 TOKENS=128 \\',
    '  KERNEL_PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json \\',
    '  bash tools/bench/eagle5_paired_bench.sh',
    '```',
    '',
    'The Colab projection is not the final benchmark. It is a candidate selector for the laptop implementation and paired bench.',
])
summary = '\n'.join(summary_lines) + '\n'
with open(f'{ARTIFACT_DIR}/summary.md', 'w') as f:
    f.write(summary)
print(summary)
